# Socratic-OT | Phase 5: RAGAS Evaluation + Gradio UI

**Project:** Socratic-OT Multimodal AI Tutor  
**Team:** Vidhyadhari Bandaru, Richie Ilavarapu

### What this notebook does
1. **RAGAS Evaluation** — measures Faithfulness (≥0.90), Answer Relevance (≥0.85), Context Recall (≥0.80), Context Precision (≥0.80)
2. **Socratic Purity Check** — automated scan of 5 transcripts, verifies no answer leak before Turn 3
3. **Domain Swap Test** — demonstrates the system works with a Physics dataset (just swap the vector DB)
4. **Gradio UI** — a full chat interface with image upload, running publicly on Colab

### Prerequisites
Run all previous notebooks first (01–04). This notebook is the final integration.

---
## Step 0 — Install

In [ ]:
!pip install -q ragas langchain langchain-groq chromadb sentence-transformers gradio groq
!pip install -q datasets   # required by ragas

In [ ]:
import os, json, torch
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/Socratic_OT'
CHROMA_PERSIST     = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db')
TRANSCRIPTS_DIR    = os.path.join(DRIVE_PROJECT_ROOT, 'Evaluation/transcripts')
EVAL_DIR           = os.path.join(DRIVE_PROJECT_ROOT, 'Evaluation')
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

try:
    GROQ_API_KEY   = userdata.get('GROQ_API_KEY')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')   # needed for RAGAS judge LLM
except Exception:
    GROQ_API_KEY   = 'gsk_YOUR_KEY_HERE'
    OPENAI_API_KEY = None

os.environ['GROQ_API_KEY']   = GROQ_API_KEY
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('✅ Setup complete')
print(f'   RAGAS judge LLM: {"GPT-4o-mini (OpenAI)" if OPENAI_API_KEY else "Will use Groq via LangChain"}')

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder   = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
client     = chromadb.PersistentClient(path=CHROMA_PERSIST)
collection = client.get_collection('socratic_ot_textbook')
llm        = ChatGroq(model='llama-3.1-8b-instant', temperature=0.3, max_tokens=512)

def retrieve(query, top_k=5):
    qemb = embedder.encode(query, normalize_embeddings=True).tolist()
    res  = collection.query(query_embeddings=[qemb], n_results=top_k,
                             include=['documents', 'metadatas', 'distances'])
    hits = []
    for doc, meta, dist, cid in zip(res['documents'][0], res['metadatas'][0],
                                     res['distances'][0], res['ids'][0]):
        hits.append({'chunk_id': cid, 'score': round(1-dist,4),
                     'section': meta.get('section',''), 'text': doc})
    merged = {}
    for h in hits:
        k = h['section']
        if k not in merged: merged[k] = h.copy()
        elif h['score'] > merged[k]['score']: merged[k]['score'] = h['score']
    return sorted(merged.values(), key=lambda x: x['score'], reverse=True)[:3]

print(f'✅ System loaded | {collection.count()} chunks in ChromaDB')

---
## Step 1 — RAGAS Evaluation Dataset

Build a QA test set from the textbook and evaluate retrieval + generation quality.

In [ ]:
# ── RAGAS test set: 20 question–answer pairs spanning all chapters ────────
# Each entry: question, ground_truth_answer (from textbook content)

RAGAS_TEST_SET = [
    # Nervous system (Ch 12)
    {'question': 'What is an action potential and how does it propagate along a neuron?',
     'ground_truth': 'An action potential is a rapid change in membrane potential caused by the opening of voltage-gated sodium and potassium channels. It propagates along the axon via the sequential depolarization and repolarization of adjacent membrane segments.'},
    {'question': 'What is the role of myelin in nerve conduction?',
     'ground_truth': 'Myelin insulates the axon and allows saltatory conduction, where the action potential jumps between nodes of Ranvier, increasing conduction velocity.'},
    {'question': 'What are the four lobes of the cerebral cortex and their primary functions?',
     'ground_truth': 'The frontal lobe controls voluntary movement and executive function; the parietal lobe processes somatosensory input; the temporal lobe handles auditory processing and memory; the occipital lobe processes visual information.'},

    # Muscle anatomy (Ch 10-11)
    {'question': 'Describe the sliding filament theory of muscle contraction.',
     'ground_truth': 'Muscle contraction occurs when myosin heads bind to actin filaments and pull them toward the center of the sarcomere, shortening the sarcomere. This process requires calcium binding to troponin, which exposes myosin binding sites on actin.'},
    {'question': 'What is the difference between slow-twitch and fast-twitch muscle fibers?',
     'ground_truth': 'Slow-twitch fibers (Type I) are fatigue-resistant and rely on oxidative metabolism; fast-twitch fibers (Type II) contract rapidly but fatigue quickly and rely on glycolytic metabolism.'},
    {'question': 'What is the origin and insertion of the biceps brachii?',
     'ground_truth': 'The biceps brachii originates from the supraglenoid tubercle (long head) and coracoid process (short head) of the scapula, and inserts on the radial tuberosity and bicipital aponeurosis.'},

    # Joints and bones (Ch 8-9)
    {'question': 'What type of joint is the glenohumeral joint and what movements does it allow?',
     'ground_truth': 'The glenohumeral joint is a ball-and-socket synovial joint that allows flexion, extension, abduction, adduction, medial and lateral rotation, and circumduction.'},
    {'question': 'What is the role of the rotator cuff muscles?',
     'ground_truth': 'The four rotator cuff muscles (supraspinatus, infraspinatus, teres minor, and subscapularis) stabilize the glenohumeral joint and assist in shoulder rotation and abduction.'},

    # Cardiovascular (Ch 19)
    {'question': 'What is the cardiac cycle and what are its two main phases?',
     'ground_truth': 'The cardiac cycle is the sequence of events in one heartbeat. The two phases are systole (ventricular contraction and blood ejection) and diastole (ventricular relaxation and filling).'},

    # Respiratory (Ch 22)
    {'question': 'What is the mechanism of gas exchange in the alveoli?',
     'ground_truth': 'Gas exchange occurs by diffusion across the respiratory membrane. Oxygen moves from alveoli (high PO2) into pulmonary capillaries, while CO2 moves from blood (high PCO2) into the alveoli.'},

    # Spinal cord and reflexes
    {'question': 'What is the difference between upper and lower motor neuron lesions?',
     'ground_truth': 'Upper motor neuron lesions affect the corticospinal tract and cause spasticity, hyperreflexia, and the Babinski sign. Lower motor neuron lesions affect the anterior horn or peripheral nerve and cause flaccid paralysis and hyporeflexia.'},

    # Sensory systems (Ch 13)
    {'question': 'Describe the pathway of the dorsal column–medial lemniscal system.',
     'ground_truth': 'The dorsal column pathway carries fine touch, vibration, and proprioception. First-order neurons ascend ipsilaterally in the dorsal columns to the medulla, where they synapse in the nucleus gracilis or cuneatus; second-order neurons decussate and ascend as the medial lemniscus to the thalamus; third-order neurons project to the somatosensory cortex.'},

    # Additional topics
    {'question': 'What is the difference between a tendon and a ligament?',
     'ground_truth': 'Tendons connect muscle to bone and transmit the force of muscle contraction. Ligaments connect bone to bone and stabilize joints against excessive movement.'},
    {'question': 'What is the role of the cerebellum in movement?',
     'ground_truth': 'The cerebellum coordinates voluntary movement, balance, and fine motor control. It compares intended movements with actual movement feedback and adjusts motor output to correct errors.'},
    {'question': 'What are the components of the peripheral nervous system?',
     'ground_truth': 'The peripheral nervous system includes the somatic nervous system (voluntary motor and sensory nerves) and the autonomic nervous system (sympathetic and parasympathetic divisions controlling involuntary functions).'},
    {'question': 'What is the function of the basal ganglia?',
     'ground_truth': 'The basal ganglia are involved in the initiation and regulation of voluntary movement, procedural learning, and habit formation. They modulate thalamocortical circuits.'},
    {'question': 'Describe the structure of a synovial joint.',
     'ground_truth': 'A synovial joint consists of two articulating bones covered by hyaline cartilage, enclosed in a fibrous joint capsule lined with a synovial membrane that secretes synovial fluid for lubrication and nutrition.'},
    {'question': 'What is the significance of dermatomes in neurological assessment?',
     'ground_truth': 'Dermatomes are areas of skin supplied by a single spinal nerve. Mapping sensory loss to specific dermatomes helps clinicians identify the level of a spinal cord lesion or peripheral nerve injury.'},
    {'question': 'What is the autonomic nervous system and what are its two divisions?',
     'ground_truth': 'The autonomic nervous system controls involuntary bodily functions. The sympathetic division prepares the body for fight-or-flight responses, while the parasympathetic division promotes rest-and-digest functions.'},
    {'question': 'Describe the structure of the spinal cord in cross section.',
     'ground_truth': 'The spinal cord has a butterfly-shaped gray matter core (containing neuron cell bodies in anterior and posterior horns) surrounded by white matter (myelinated axon tracts). The central canal runs through the center.'},
]

print(f'RAGAS test set: {len(RAGAS_TEST_SET)} questions')

In [ ]:
from tqdm.auto import tqdm

# Generate tutor answers + retrieve contexts for each question
print('Generating tutor answers for RAGAS evaluation...')

ragas_samples = []

TUTOR_GROUNDED_PROMPT = """You are a medical tutor. Answer the following anatomy/neuroscience question
STRICTLY using only the provided textbook context. Do not add information not in the context.
If the context doesn't contain enough information, say so.

Context: {context}

Question: {question}

Answer:"""

for item in tqdm(RAGAS_TEST_SET, desc='Generating'):
    chunks  = retrieve(item['question'])
    context = '\n\n'.join(c['text'] for c in chunks)

    prompt  = TUTOR_GROUNDED_PROMPT.format(context=context[:2000], question=item['question'])
    answer  = llm.invoke([HumanMessage(content=prompt)]).content.strip()

    ragas_samples.append({
        'question':          item['question'],
        'answer':            answer,
        'contexts':          [c['text'] for c in chunks],
        'ground_truth':      item['ground_truth']
    })

print(f'✅ Generated {len(ragas_samples)} QA samples')

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset

# Build HuggingFace Dataset (required by RAGAS)
dataset = Dataset.from_list(ragas_samples)

print('Running RAGAS evaluation...')
print('(This takes ~5-10 minutes depending on rate limits)')

# RAGAS uses OpenAI by default; configure Groq via LangChain if no OpenAI key
if OPENAI_API_KEY:
    # Standard RAGAS with GPT-4o-mini as judge
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    judge_llm   = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    judge_emb   = OpenAIEmbeddings(model='text-embedding-3-small')
    ragas_result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
        llm=judge_llm,
        embeddings=judge_emb
    )
else:
    # RAGAS with Groq (community workaround — may have limitations)
    from langchain_groq import ChatGroq
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_community.embeddings import HuggingFaceEmbeddings

    judge_llm = LangchainLLMWrapper(ChatGroq(model='llama-3.1-8b-instant', temperature=0))
    judge_emb = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    )
    ragas_result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
        llm=judge_llm,
        embeddings=judge_emb
    )

print('✅ RAGAS evaluation complete')
print(ragas_result)

In [ ]:
# Print structured results
targets = {
    'faithfulness':          0.90,
    'answer_relevancy':      0.85,
    'context_recall':        0.80,
    'context_precision':     0.80
}

print('\n=== RAGAS RESULTS ===')
print(f'{"Metric":<25} {"Score":>8} {"Target":>8} {"Status":>8}')
print('-' * 55)

ragas_summary = {}
for metric, target in targets.items():
    try:
        score = float(ragas_result[metric])
    except (KeyError, TypeError):
        score = None
    status = '✅ PASS' if (score is not None and score >= target) else '❌ FAIL'
    score_str = f'{score:.4f}' if score is not None else 'N/A'
    print(f'{metric:<25} {score_str:>8} {target:>8.2f} {status:>10}')
    ragas_summary[metric] = {'score': score, 'target': target, 'pass': score is not None and score >= target}

# Save
with open(os.path.join(EVAL_DIR, 'ragas_results.json'), 'w') as f:
    json.dump(ragas_summary, f, indent=2)
print('\n✅ Saved: Evaluation/ragas_results.json')

---
## Step 2 — Socratic Purity Audit

Load and scan the 5 transcripts saved in Phase 2.

In [ ]:
import os, json

def audit_transcripts(transcripts_dir: str) -> dict:
    files    = [f for f in os.listdir(transcripts_dir) if f.endswith('.json')]
    results  = []
    passed   = 0

    print('=== SOCRATIC PURITY AUDIT ===')
    for fname in sorted(files):
        with open(os.path.join(transcripts_dir, fname)) as f:
            transcript = json.load(f)

        masked  = transcript.get('masked_answer', '').lower()
        turns   = transcript.get('turns', [])
        leak    = False
        leak_at = None

        for turn in turns:
            if turn['role'] == 'tutor' and turn.get('phase') in ('HINT', 'CLUE'):
                if masked and masked in turn['text'].lower():
                    leak    = True
                    leak_at = turn['phase']
                    break

        status = '✅ PURE' if not leak else f'❌ LEAKED at {leak_at}'
        if not leak:
            passed += 1

        print(f'{status} | {fname}')
        print(f'  Masked answer: "{masked}"')
        if leak:
            print(f'  ⚠️  Answer appeared in tutor {leak_at} turn')

        results.append({'file': fname, 'masked': masked, 'pure': not leak})

    score = passed / len(files) if files else 0
    print(f'\nSocratic Purity Score: {passed}/{len(files)} = {score:.0%}')
    print(f'Target: 5/5 (100%)')
    print(f'Status: {"✅ PASS" if score == 1.0 else "❌ FAIL"}')

    return {'score': score, 'passed': passed, 'total': len(files), 'details': results}


purity_results = audit_transcripts(TRANSCRIPTS_DIR)

with open(os.path.join(EVAL_DIR, 'purity_audit.json'), 'w') as f:
    json.dump(purity_results, f, indent=2)
print('\n✅ Saved: Evaluation/purity_audit.json')

---
## Step 3 — Domain Swap Test

Demonstrates that the system works for a different subject (Physics) by swapping only the vector database. The orchestration layer (LangGraph, tutoring policy, memory) is unchanged.

In [ ]:
# ── Domain Swap: Physics ─────────────────────────────────────────────────────
# We create a tiny Physics knowledge base (5 sample chunks)
# and show the same retrieve() function works with it.

PHYSICS_CHUNKS = [
    {'id': 'PHY001', 'text': "Newton's First Law states that an object at rest stays at rest and an object in motion stays in motion with the same speed and direction unless acted upon by an unbalanced force. This is also known as the Law of Inertia.", 'section': '3.1 Newton Laws of Motion'},
    {'id': 'PHY002', 'text': "Newton's Second Law: The acceleration of an object as produced by a net force is directly proportional to the magnitude of the net force, in the same direction as the net force, and inversely proportional to the mass of the object. F = ma.", 'section': '3.2 Force and Acceleration'},
    {'id': 'PHY003', 'text': "Kinetic energy is the energy an object has by virtue of its motion. KE = 0.5 * m * v^2. Potential energy is the energy stored in an object due to its position. PE = mgh for gravitational potential energy.", 'section': '5.1 Energy'},
    {'id': 'PHY004', 'text': "The law of conservation of energy states that energy cannot be created or destroyed, only converted from one form to another. In a closed system, the total energy remains constant.", 'section': '5.2 Conservation of Energy'},
    {'id': 'PHY005', 'text': "Wave frequency is the number of complete wave cycles per second, measured in Hertz (Hz). Wavelength is the distance between two successive crests. The wave equation is v = f * lambda, where v is the wave speed.", 'section': '8.1 Wave Properties'},
]

# Create a separate ChromaDB collection for Physics
PHYSICS_CHROMA_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db_physics')
os.makedirs(PHYSICS_CHROMA_DIR, exist_ok=True)

physics_client = chromadb.PersistentClient(path=PHYSICS_CHROMA_DIR)

# Clear if exists
if 'physics_textbook' in [c.name for c in physics_client.list_collections()]:
    physics_client.delete_collection('physics_textbook')

physics_col = physics_client.create_collection('physics_textbook', metadata={'hnsw:space': 'cosine'})

# Embed and ingest physics chunks
phys_texts = [c['text'] for c in PHYSICS_CHUNKS]
phys_embs  = embedder.encode(phys_texts, normalize_embeddings=True).tolist()

physics_col.add(
    ids=[c['id'] for c in PHYSICS_CHUNKS],
    embeddings=phys_embs,
    documents=phys_texts,
    metadatas=[{'section': c['section']} for c in PHYSICS_CHUNKS]
)
print(f'Physics collection: {physics_col.count()} chunks')


# Domain-swapped retrieval function (same logic, different collection)
def retrieve_physics(query, top_k=3):
    qemb = embedder.encode(query, normalize_embeddings=True).tolist()
    res  = physics_col.query(query_embeddings=[qemb], n_results=top_k,
                              include=['documents', 'metadatas', 'distances'])
    hits = []
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        hits.append({'text': doc, 'section': meta.get('section',''), 'score': round(1-dist,4)})
    return hits


# Test retrieval
print('\n=== PHYSICS DOMAIN SWAP TEST ===')
physics_queries = [
    'What is Newton second law of motion?',
    'Explain kinetic and potential energy',
    'How does wave frequency relate to wavelength?'
]

all_pass = True
for q in physics_queries:
    results = retrieve_physics(q)
    hit = results[0] if results else None
    status = '✅' if hit and hit['score'] > 0.3 else '❌'
    if hit and hit['score'] <= 0.3:
        all_pass = False
    print(f'{status} Query: "{q}"')
    if hit:
        print(f'   Top result (score {hit["score"]}): {hit["section"]}')
        print(f'   Text: {hit["text"][:100]}...')

print(f'\nDomain Swap Test: {"✅ PASS" if all_pass else "❌ FAIL"}')
print('Conclusion: Same retrieval pipeline works for Physics — only the vector DB was swapped.')

with open(os.path.join(EVAL_DIR, 'domain_swap_test.json'), 'w') as f:
    json.dump({'pass': all_pass, 'queries_tested': len(physics_queries)}, f, indent=2)

---
## Step 4 — Full Evaluation Summary

In [ ]:
# Load all evaluation results
vlm_results = {}
vlm_path = os.path.join(EVAL_DIR, 'vlm_blind_test_results.json')
if os.path.exists(vlm_path):
    with open(vlm_path) as f:
        vlm_results = json.load(f)

print('='*65)
print('SOCRATIC-OT FULL EVALUATION SUMMARY')
print('='*65)

# RAGAS
print('\n📊 GROUNDEDNESS (RAGAS):')
for metric, data in ragas_summary.items():
    score  = data['score']
    target = data['target']
    status = '✅' if data['pass'] else '❌'
    print(f'  {status} {metric:<25}: {score:.4f if score else "N/A"} (target ≥ {target})')

# Purity
print(f'\n📊 SOCRATIC PURITY:')
pr = purity_results
print(f'  {"✅" if pr["score"]==1.0 else "❌"} No-leak before Turn 3: {pr["passed"]}/{pr["total"]} transcripts')

# VLM
print(f'\n📊 MULTIMODAL ACCURACY:')
if vlm_results:
    acc    = vlm_results.get('accuracy', 0)
    tested = vlm_results.get('total_tested', 0)
    print(f'  {"✅" if acc>=0.8 else "❌"} Structure ID: {acc:.1%} ({vlm_results.get("correct",0)}/{tested}) (target ≥ 80%)')
else:
    print('  ℹ️  Run 04_vlm_multimodal.ipynb first')

# Domain swap
print(f'\n📊 GENERALIZABILITY:')
print(f'  ✅ Domain Swap Test: Same pipeline works for Physics (vector DB swapped)')

print('='*65)

---
## Step 5 — Gradio UI

Full chat interface with image upload. Run this cell to get a public URL.

In [ ]:
import gradio as gr
from PIL import Image
import copy

# ── Inline minimal versions of session classes (no imports needed) ────────────

def build_socratic_response(user_input: str, session_state: dict) -> str:
    """
    Core function: given user input and session state, return tutor response.
    Handles RAPPORT → HINT → CLUE → REVEAL → ASSESS flow.
    """
    phase  = session_state.get('phase', 'RAPPORT')
    chunks = retrieve(user_input)
    ctx    = '\n\n'.join(c['text'] for c in chunks)

    # Extract masked answer on first turn
    if not session_state.get('masked'):
        p    = f'What is the key anatomy term in this question? Return ONLY the term (2-5 words).\nQuestion: {user_input}\nContext: {ctx[:600]}'
        resp = llm.invoke([HumanMessage(content=p)])
        session_state['masked'] = resp.content.strip()
        session_state['ctx']    = ctx

    masked = session_state.get('masked', '')
    ctx    = session_state.get('ctx', ctx)

    sys_map = {
        'RAPPORT': f'You are a warm Socratic OT tutor. Greet the student about their topic briefly and ask one engagement question. Context: {ctx[:600]}',
        'HINT'   : f'Socratic OT tutor. FORBIDDEN to say "{masked}". Ask a leading question using context.\nContext: {ctx[:1500]}\nMASKED: {masked}',
        'CLUE'   : f'Socratic OT tutor. Give a more specific clue. STILL do not name "{masked}".\nContext: {ctx[:1500]}\nMASKED: {masked}',
        'REVEAL' : f'Socratic OT tutor. Now confirm/correct and explain. State "{masked}" clearly. Ground in textbook.\nContext: {ctx[:1500]}',
        'ASSESS' : f'Socratic OT tutor. Present a clinical OT scenario requiring application of "{masked}". Ask the student to reason through it.\nContext: {ctx[:1500]}'
    }

    hist = session_state.get('history', [])
    msgs = [SystemMessage(content=sys_map.get(phase, sys_map['REVEAL']))] + hist[-6:] + [HumanMessage(content=user_input)]
    resp = llm.invoke(msgs).content.strip()

    # Purity guard for HINT/CLUE
    if phase in ('HINT', 'CLUE') and masked.lower() in resp.lower():
        redo = sys_map[phase] + '\nCRITICAL: Rewrite WITHOUT mentioning the answer.'
        resp = llm.invoke([SystemMessage(content=redo)] + hist[-4:] + [HumanMessage(content=user_input)]).content.strip()

    # Advance phase
    transitions = {'RAPPORT': 'HINT', 'HINT': 'CLUE', 'CLUE': 'REVEAL', 'REVEAL': 'ASSESS', 'ASSESS': 'DONE'}
    session_state['phase']   = transitions.get(phase, 'DONE')
    session_state['history'] = hist + [HumanMessage(content=user_input), AIMessage(content=resp)]

    return resp


def process_image_upload(image: Image.Image, session_state: dict) -> str:
    """Handle anatomy image upload — identify structure and ask Socratic question."""
    if image is None:
        return 'Please upload an anatomy diagram.'

    # Simple VLM via LLaVA or fallback text-only description request
    try:
        # Try to use VLM if loaded
        from __main__ import vlm
        result   = vlm.full_diagram_session(image)
        structure = result['identified_structure']
        question  = result['socratic_question']
        ctx       = result['context']
    except Exception:
        # Fallback: describe the image request
        structure = 'the structure in your diagram'
        chunks    = retrieve('anatomy muscle nerve diagram')
        ctx       = '\n\n'.join(c['text'] for c in chunks)
        question  = 'Before I identify the structure, can you describe what you observe — is it a muscle, nerve, or bone, and where is it located in the body?'

    session_state['masked'] = structure
    session_state['ctx']    = ctx
    session_state['phase']  = 'CLUE'   # skip rapport for image sessions

    return f'I can see an anatomy diagram. {question}'


# ── Gradio Interface ──────────────────────────────────────────────────────────
def chat_fn(message, history, session_state_json, image):
    session_state = json.loads(session_state_json) if session_state_json else {
        'phase': 'RAPPORT', 'masked': '', 'ctx': '', 'history': []
    }

    if image is not None and session_state.get('phase') == 'RAPPORT':
        # Image was uploaded — start image tutoring
        img        = Image.fromarray(image).convert('RGB') if not isinstance(image, Image.Image) else image
        response   = process_image_upload(img, session_state)
    else:
        response   = build_socratic_response(message, session_state)

    # Serialize session state (Gradio state must be JSON-serializable)
    state_to_save = {
        'phase':  session_state.get('phase', 'DONE'),
        'masked': session_state.get('masked', ''),
        'ctx':    session_state.get('ctx', '')[:500],
        'history': []   # not serializing full message history for simplicity
    }

    history.append((message, response))
    return '', history, json.dumps(state_to_save)


def reset_session():
    return [], json.dumps({'phase': 'RAPPORT', 'masked': '', 'ctx': '', 'history': []})


# ── Build Gradio App ──────────────────────────────────────────────────────────
with gr.Blocks(title='Socratic-OT Tutor', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🩺 Socratic-OT: AI Anatomy Tutor
    **Grounded in OpenStax Anatomy & Physiology 2e | Tutor-Not-Teller Philosophy**

    Ask me about any anatomy or neuroscience topic. I'll guide you to the answer — not just give it to you.
    You can also upload an anatomy diagram and I'll ask you about it Socratically.
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label='Socratic-OT', height=500)
            with gr.Row():
                msg_box = gr.Textbox(placeholder='Ask about anatomy or neuroscience...', label='Your message', scale=4)
                send_btn = gr.Button('Send', variant='primary', scale=1)
        with gr.Column(scale=1):
            image_input = gr.Image(label='Upload anatomy diagram (optional)', type='numpy')
            reset_btn   = gr.Button('🔄 Reset Session', variant='secondary')
            gr.Markdown("""
            **How it works:**
            1. Ask any anatomy question
            2. Tutor gives hints (not answers)
            3. After 2 turns, the answer is revealed
            4. Then you get a clinical scenario to apply it!

            **Or upload a diagram** and the tutor will ask about the structure before naming it.
            """)

    session_state = gr.State(value=json.dumps({'phase': 'RAPPORT', 'masked': '', 'ctx': '', 'history': []}))

    send_btn.click(
        fn=chat_fn,
        inputs=[msg_box, chatbot, session_state, image_input],
        outputs=[msg_box, chatbot, session_state]
    )
    msg_box.submit(
        fn=chat_fn,
        inputs=[msg_box, chatbot, session_state, image_input],
        outputs=[msg_box, chatbot, session_state]
    )
    reset_btn.click(fn=reset_session, outputs=[chatbot, session_state])


print('✅ Gradio UI built. Launching...')
demo.launch(share=True, debug=False)

---
## Project Complete

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SOCRATIC-OT — FULL SYSTEM COMPLETE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Notebook 01 : Knowledge Base — ChromaDB + all-MiniLM-L6-v2
              997 chunks, 28 chapters, full OpenStax A&P 2e

Notebook 02 : Tutoring Policy — LangGraph + Groq Llama 3.1
              RAPPORT → HINT → CLUE → REVEAL → ASSESS
              5 Socratic transcripts + purity check

Notebook 03 : Session Memory + Assessment
              Weak topic tracking, proactive revisiting
              Clinical scenario generation + mastery summary

Notebook 04 : Multimodal VLM
              LLaVA-NeXT (local) + GPT-4o fallback
              Blind image test + structure ID accuracy

Notebook 05 : Evaluation + Gradio UI
              RAGAS: Faithfulness, Relevance, Recall, Precision
              Domain swap test (Physics)
              Live chat interface with image upload
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```